# Session 4 (Day 2 — Afternoon)
## Macroeconomic Data & Market Context

**GIS Executive Seminar — Enhancing Data Analysis Skills for Informed Investment Insights**

---
> *Guiding question: "What is the macroeconomic environment telling us — and what does it mean for portfolio positioning?"*

### Learning Objectives
By the end of this session you will be able to:
- Download and interpret yield curve data from FRED
- Identify the 2Y/10Y spread and understand its historical relationship with recessions
- Compute headline vs. core CPI and convert nominal returns to real returns
- Understand FX dynamics and how a currency move affects the USD return of a foreign asset
- Assemble a macro dashboard and apply a simple regime classification rule

### How to Use This Notebook
Each cell is pre-written. Run them in order with Shift+Enter. The data downloads fresh at the top — no files needed from previous sessions.

## Setup

This notebook re-downloads macro data from FRED and price data from yfinance fresh each time it runs. This takes about 30 seconds. Every session is self-contained so you can run any day independently without relying on files saved from previous sessions.

**Before running:** if you are on Google Colab, make sure you are connected to a runtime (green circle, top right). Then run each cell in order with **Shift+Enter**.

In [ ]:
# Install all libraries needed for this session
# fredapi: the official Python client for the Federal Reserve Economic Data API
# yfinance: stock and ETF price data from Yahoo Finance

!pip install yfinance fredapi  --quiet

print('Libraries installed.')

In [ ]:
import pandas as pd             # pandas: DataFrames — the spreadsheet engine
import numpy as np              # numpy: numerical computing — arrays, maths functions
import matplotlib.pyplot as plt  # matplotlib: static charts — the core plotting library
import matplotlib.ticker as mticker   # ticker: fine-grained axis formatting for charts
from fredapi import Fred        # fredapi: Federal Reserve macro data downloader
import yfinance as yf           # yfinance: stock and ETF price data from Yahoo Finance
from datetime import date       # date: today's date without time component

# Set the default chart style — 'seaborn-v0_8-whitegrid' gives clean white backgrounds
plt.style.use('seaborn-v0_8-whitegrid')

print('All libraries imported.')

In [ ]:
# ← CHANGE ME ─────────────────────────────────────────────────────────────────
# To get a free FRED API key:
#   1. Go to https://fred.stlouisfed.org/
#   2. Click 'My Account' → 'Create an Account'
#   3. After registering, go to 'My Account' → 'API Keys' → 'Request API Key'
#   4. Copy the key (a long string of letters and numbers) and paste it below

FRED_API_KEY = 'YOUR_FRED_API_KEY'
fred = Fred(api_key=FRED_API_KEY)   # initialise the FRED client with the API key
print('FRED client ready.')

In [ ]:
# ── DATE RANGE: 5 YEARS ────────────────────────────────────────────────────────
# We use 5 years for macro data — a longer horizon than for prices.
# Why 5 years? The yield curve typically completes a full cycle (normal → flat → inverted
# → normal again) over 3–7 years. 5 years gives enough history to see at least one
# full cycle and any recent recession/recovery.

end_date   = date.today()   # today — the most recent data available

start_date = date(
    end_date.year - 5,   # go back 5 years
    end_date.month,      # same month
    end_date.day,        # same day
)

print(f'Macro data window: {start_date}  →  {end_date}  (5 years)')
print()
print('This window will capture:')
print('  - At least one complete rate cycle (hike + cut or cut + hike)')
print('  - Any recent yield curve inversions')
print('  - At least 60 monthly CPI data points')
print('  - Multiple FX trend episodes')

---
## Section 1: The Yield Curve

### What is the yield curve?

The yield curve is a **snapshot of interest rates across maturities** — from 1 month to 30 years — at a single point in time. It shows the annual return you would earn by lending to the US government for different lengths of time.

Think of it as a term structure: a row of Bloomberg `T <Maturity> <Govt>` pages displayed side by side.

### The three shapes and what they mean

**Normal (upward sloping):** Short-term rates are lower than long-term rates. This is the default state — lenders demand a higher return for tying up their money for longer. It signals a healthy economy where investors expect growth to continue.

**Flat:** Short and long rates are close together. Often seen in late economic cycles, when the central bank has raised short rates but markets expect slower growth (and therefore lower future rates). A flat curve compresses bank net interest margins — bad for financials.

**Inverted:** Short-term rates are **above** long-term rates. Historically, this has been the most reliable leading indicator of recession. Every US recession since 1970 has been preceded by a yield curve inversion. The lag is typically 6–18 months.

**Why does inversion predict recession?**  
The mechanism: the Federal Reserve raises short-term rates aggressively to cool inflation. Long-term rates fall because bond markets look further ahead and price in the coming recession and subsequent rate cuts. The two rates cross — the curve inverts — signalling the market's collective belief that growth is about to slow sharply.

### Duration: a plain-English explanation

Duration measures **how sensitive a bond's price is to a change in interest rates**. Think of it as a lever:
- A short-term bond (1-year maturity) has a short lever — a 1% rise in rates moves its price very little
- A long-term bond (30-year maturity) has a long lever — a 1% rise in rates can drop its price by 20%+

This is why, when the yield curve is inverted (short rates higher than long), investors holding long-term bonds face serious mark-to-market losses if rates continue to rise.

### Investment implications of yield curve positioning

| Curve Shape | Bond Positioning | Equity Sector Preference |
|---|---|---|
| Normal | Favour longer maturities for yield | Cyclicals, Financials |
| Flat | Neutral; watch for turning point | Quality growth, Defensives |
| Inverted | Favour short maturities | Healthcare, Consumer Staples, Utilities |

In [ ]:
# ── YIELD CURVE DATA FROM FRED ────────────────────────────────────────────────
# We download constant-maturity Treasury yields at 8 maturities.
# Together they form the yield curve — a snapshot of rates across the time spectrum.

# Dictionary mapping a plain-English maturity label to its FRED series ID
YIELD_CURVE_SERIES = {
    '1M' : 'GS1M',    # 1-month Treasury yield
    '3M' : 'GS3M',    # 3-month Treasury yield
    '6M' : 'GS6M',    # 6-month Treasury yield
    '1Y' : 'GS1',     # 1-year Treasury yield
    '2Y' : 'GS2',     # 2-year Treasury yield  ← key: sensitive to Fed rate expectations
    '5Y' : 'GS5',     # 5-year Treasury yield
    '10Y': 'GS10',    # 10-year Treasury yield ← key: the benchmark long rate
    '30Y': 'GS30',    # 30-year Treasury yield
}

# Download each series and collect into a dictionary
yield_data = {}   # empty dictionary — we will fill it one series at a time
for label, series_id in YIELD_CURVE_SERIES.items():   # loop over each maturity
    yield_data[label] = fred.get_series(   # download this maturity's yield history
        series_id,
        observation_start=start_date,   # beginning of our 5-year window
        observation_end=end_date,       # up to today
    )
    print(f'  Downloaded: {label} ({series_id})')   # progress indicator

# Combine into one DataFrame — rows=dates, columns=maturities
yield_curve_df = (
    pd.DataFrame(yield_data)    # convert the dictionary of series into a DataFrame
    .dropna()                   # remove rows with any missing values (some maturities report monthly)
)

print(f'\nYield curve DataFrame: {yield_curve_df.shape[0]} observations')
print(f'Date range: {yield_curve_df.index[0].date()}  →  {yield_curve_df.index[-1].date()}')
print()
print('Most recent 3 rows:')
print(yield_curve_df.tail(3).round(2).to_string())

In [ ]:
# ── TODAY'S YIELD CURVE SNAPSHOT ─────────────────────────────────────────────
# Take the most recent observation from the yield curve DataFrame.
# This is a cross-sectional snapshot: rates at each maturity as of the latest data date.

latest_curve = yield_curve_df.iloc[-1]    # most recent row — yields at all 8 maturities
latest_date  = yield_curve_df.index[-1].date()   # the date of that most recent observation

# Maturity labels for the x-axis
maturity_labels  = list(YIELD_CURVE_SERIES.keys())   # ['1M', '3M', '6M', '1Y', '2Y', '5Y', '10Y', '30Y']

fig, ax = plt.subplots(figsize=(11, 5))

# Bar chart: one bar per maturity, height = yield
ax.bar(
    maturity_labels,         # x-axis: maturity labels
    latest_curve.values,     # y-axis: yield at each maturity
    color='steelblue',
    alpha=0.7,
    edgecolor='white',
    width=0.6,
    label='Treasury Yield (%)',
)

# Overlay a line connecting the bars — this is the 'curve' shape
ax.plot(
    maturity_labels,
    latest_curve.values,
    color='steelblue',
    linewidth=2,
    marker='o',    # circle marker at each data point
    markersize=6,
)

# Extract the 2Y and 10Y rates for reference lines and spread calculation
rate_2y  = latest_curve['2Y']    # 2-year Treasury yield — most sensitive to Fed actions
rate_10y = latest_curve['10Y']   # 10-year Treasury yield — the benchmark long rate
spread   = rate_10y - rate_2y    # 2Y/10Y spread: positive = normal, negative = inverted

# Add horizontal dashed reference lines for 2Y and 10Y
ax.axhline(rate_2y,  color='orange', linewidth=1, linestyle='--', alpha=0.7, label=f'2Y = {rate_2y:.2f}%')
ax.axhline(rate_10y, color='seagreen',  linewidth=1, linestyle='--', alpha=0.7, label=f'10Y = {rate_10y:.2f}%')

# Label each bar with its yield value — printed just above the bar top
for label, rate in zip(maturity_labels, latest_curve.values):
    ax.text(label, rate + 0.05, f'{rate:.2f}%', ha='center', va='bottom', fontsize=9)

ax.set_title(
    f'US Treasury Yield Curve — {latest_date}',
    fontsize=14, fontweight='bold',
)
ax.set_xlabel('Maturity', fontsize=12)
ax.set_ylabel('Yield (%)', fontsize=12)
ax.legend(fontsize=10)
ax.grid(True, alpha=0.3, axis='y')   # horizontal gridlines only — cleaner look

# Classify the curve shape based on the 2Y/10Y spread
curve_shape = 'INVERTED' if spread < 0 else 'NORMAL' if spread > 0.5 else 'FLAT'

plt.tight_layout()
plt.show()

print()
print(f'2Y yield:   {rate_2y:.2f}%')
print(f'10Y yield:  {rate_10y:.2f}%')
print(f'2Y/10Y spread: {spread:+.2f} percentage points')
print(f'Curve shape: {curve_shape}')

In [ ]:
# ── 2Y/10Y YIELD CURVE SPREAD OVER TIME ───────────────────────────────────────
# T10Y2Y is the difference between the 10-year and 2-year Treasury yields.
# When T10Y2Y < 0, the yield curve is inverted.
# Inversions have preceded every US recession since 1970.

spread_series = fred.get_series(
    'T10Y2Y',                         # FRED series ID for the 10Y-2Y spread
    observation_start=start_date,     # beginning of our 5-year window
    observation_end=end_date,         # up to today
)

print(f'T10Y2Y series: {len(spread_series)} observations')
print(f'Current value: {spread_series.dropna().iloc[-1]:+.2f} percentage points')
print(f'Has been negative (inverted): {(spread_series < 0).sum()} days out of {len(spread_series)}')
print()
print('T10Y2Y interpretation:')
print('  > 0:  Normal curve (long rates above short rates)')
print('  = 0:  Flat curve (short and long rates equal)')
print('  < 0:  Inverted curve (short rates above long rates — recession warning)')

In [ ]:
# ── RECESSION SHADING ─────────────────────────────────────────────────────────
# USREC from FRED: 1 = the US economy is in an NBER-designated recession, 0 = expansion
# We shade the recession periods grey on the spread chart to see the historical relationship.

usrec = fred.get_series(
    'USREC',                         # NBER recession indicator (monthly)
    observation_start=start_date,
    observation_end=end_date,
)

# Find the START dates of each recession: days where the indicator changes from 0 → 1
rec_starts = usrec[
    (usrec == 1) & (usrec.shift(1) == 0)    # today = recession, yesterday = expansion
].index

# Find the END dates: days where the indicator changes from 1 → 0
rec_ends = usrec[
    (usrec == 0) & (usrec.shift(1) == 1)    # today = expansion, yesterday = recession
].index

print(f'Recession periods found in the data window: {len(rec_starts)}')
if len(rec_starts) > 0:   # only print if there are recession periods in the window
    for s, e in zip(rec_starts, rec_ends):
        print(f'  {s.date()}  →  {e.date()}')
else:
    print('  No completed recession periods in the 5-year window.')
    print('  (Recession shading will not appear on the chart below.)')

In [ ]:
# ── 2Y/10Y SPREAD CHART WITH RECESSION SHADING ────────────────────────────────
fig, ax = plt.subplots(figsize=(14, 5))

# Plot the spread time series
ax.plot(
    spread_series.index,
    spread_series.values,
    color='steelblue',
    linewidth=1.8,
    label='2Y/10Y Spread (percentage points)',
)

# Shade the area below zero (inversions) — the warning zone
ax.fill_between(
    spread_series.index,
    spread_series.values,
    0,                           # fill between the line and zero
    where=spread_series < 0,     # only shade where the spread is negative (inverted)
    color='crimson',
    alpha=0.3,
    label='Inverted (< 0)',
)

# Add horizontal zero reference line
ax.axhline(0, color='black', linewidth=1, linestyle='-')   # clear boundary at zero

# Shade recession periods in grey
labeled_recession = False    # track whether we have added the legend entry yet
for s, e in zip(rec_starts, rec_ends):
    ax.axvspan(
        s, e,                           # start and end of the recession period
        alpha=0.2,
        color='grey',
        label='NBER Recession' if not labeled_recession else '',   # label only once
    )
    labeled_recession = True   # don't add the label again for subsequent recessions

ax.set_title(
    '2Y/10Y Treasury Yield Spread with NBER Recession Shading',
    fontsize=14, fontweight='bold',
)
ax.set_xlabel('Date', fontsize=12)
ax.set_ylabel('Spread (percentage points)', fontsize=12)
ax.legend(fontsize=10)
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print()
print('Key observation: inversions (shaded in red) have preceded recessions (grey).')
print('Note the lag: recession typically begins 6–18 months after the inversion starts.')
print()
print('Discussion: "Given today\'s curve shape, what asset class positioning does')
print('            the historical evidence suggest? What would push you to deviate?"')

In [ ]:
# ── YIELD CURVE SUMMARY ────────────────────────────────────────────────────────
# Capture the current yield curve state in a single dictionary for use in the dashboard later

yield_summary = {
    'latest_date'  : str(latest_date),         # most recent observation date
    'rate_2y'      : round(rate_2y, 4),         # 2-year yield
    'rate_10y'     : round(rate_10y, 4),        # 10-year yield
    'spread_2y10y' : round(spread, 4),          # 2Y/10Y spread
    'curve_shape'  : curve_shape,               # NORMAL / FLAT / INVERTED
    'spread_current': round(spread_series.dropna().iloc[-1], 4),   # latest T10Y2Y from FRED
}

print('=== YIELD CURVE SUMMARY ===')
for key, val in yield_summary.items():
    print(f'  {key:<22} {val}')

# Optional export — uncomment to save
# yield_curve_df.to_csv('gis_yield_curve_tracker.csv')   # ← uncomment to save

print()
print('Optional: uncomment the line above to save the full yield curve history to CSV.')

---
## Section 2: Inflation & Real Returns

### What does CPI measure?

The Consumer Price Index (CPI) measures the average change over time in the prices paid by urban consumers for a **representative basket** of goods and services. The US Bureau of Labor Statistics (BLS) surveys thousands of prices every month — groceries, rent, gasoline, healthcare, clothing, entertainment — and aggregates them into a single index.

In Bloomberg, the headline CPI is tracked on `CPI YOY Index`. In FRED, the series is `CPIAUCSL`.

### Headline vs. Core CPI

**Headline CPI** includes all items — food and energy included. Energy prices (especially oil) are **volatile**: they can swing 20–30% in a year based on geopolitical events, OPEC decisions, or demand shocks. Food prices can spike with droughts or supply chain disruptions. This volatility makes headline CPI a noisy signal of underlying inflation trends.

**Core CPI** excludes food and energy — it measures the "underlying" or "sticky" component of inflation. The Federal Reserve focuses on core (or more recently, PCE core) because:
1. It better reflects the inflation the Fed can actually influence through interest rates
2. It is more persistent — once core CPI rises, it tends to stay elevated longer
3. Monetary policy works with a 12–18 month lag — so the Fed needs a forward-looking signal, not one driven by last week's oil price

### Year-over-year rate

We express CPI as a **year-over-year percentage change** (this month vs. same month last year). This removes seasonality — December is always more expensive than August, but that is not inflation; it is a seasonal pattern. YoY comparison automatically cancels these seasonal effects.

### Why real returns matter for institutional investors

**Nominal return = what your portfolio gained in money terms.**  
**Real return = what your portfolio gained in purchasing power terms.**

A portfolio nominally up 5% in a year when inflation is 8% has a **negative real return of approximately -3%**. The investor is worse off — they can buy less with their money than when they started.

For GIS and other institutional investors, the mandate is almost always expressed in real terms: *"preserve capital in real terms"* or *"target CPI + 2% per year."* Failing to beat inflation is a failure, even if the nominal return looks positive.

**In an investment committee presentation:** always show both nominal and real performance. The gap is the inflation drag — and if the drag is large, it warrants a discussion about inflation-linked assets (TIPS, real estate, commodities, infrastructure).

In [ ]:
# ── CPI: HEADLINE AND CORE ────────────────────────────────────────────────────
# CPIAUCSL: All Urban Consumers, All Items (headline CPI)
# CPILFESL: All Urban Consumers, All Items Less Food & Energy (core CPI)
# Both are monthly index levels — we compute the year-over-year % change

cpi_headline = fred.get_series(
    'CPIAUCSL',                       # headline CPI series
    observation_start=start_date,
    observation_end=end_date,
)

cpi_core = fred.get_series(
    'CPILFESL',                       # core CPI series (ex-food and energy)
    observation_start=start_date,
    observation_end=end_date,
)

# Year-over-year rate: (this month / same month last year) − 1
# .pct_change(12) computes the change from 12 periods ago (12 months = 1 year)
cpi_headline_yoy = (
    cpi_headline
    .pct_change(12)    # compare each month to the same month 12 months earlier
    .dropna()          # first 12 months will be NaN — drop them
    .mul(100)          # convert from decimal to percentage (0.034 → 3.4)
)

cpi_core_yoy = (
    cpi_core
    .pct_change(12)    # same year-over-year calculation for core CPI
    .dropna()
    .mul(100)
)

print('=== CPI YEAR-OVER-YEAR ===')
print(f'Latest headline CPI (YoY): {cpi_headline_yoy.iloc[-1]:.2f}%  as of {cpi_headline_yoy.index[-1].date()}')
print(f'Latest core CPI (YoY):     {cpi_core_yoy.iloc[-1]:.2f}%  as of {cpi_core_yoy.index[-1].date()}')
print(f'Spread (headline − core):  {cpi_headline_yoy.iloc[-1] - cpi_core_yoy.iloc[-1]:.2f}%')
print()
print('A large headline-core gap = food or energy prices driving current inflation.')
print('A narrow gap = inflation is broad-based across the economy.')

In [ ]:
# ── HEADLINE vs CORE CPI CHART WITH FED TARGET ────────────────────────────────
fig, ax = plt.subplots(figsize=(14, 5))

# Headline CPI in red — volatile, includes food and energy
ax.plot(
    cpi_headline_yoy.index,
    cpi_headline_yoy.values,
    color='crimson',
    linewidth=2,
    label='Headline CPI (YoY %)',
)

# Core CPI in blue — smoother, excludes food and energy
ax.plot(
    cpi_core_yoy.index,
    cpi_core_yoy.values,
    color='steelblue',
    linewidth=2,
    linestyle='--',
    label='Core CPI — ex-food & energy (YoY %)',
)

# The Fed's 2% inflation target — a horizontal reference line
ax.axhline(
    2.0,
    color='seagreen',
    linewidth=1.5,
    linestyle=':',
    label='Fed 2% target',
)

# Shade recession periods in grey (if any in the 5-year window)
labeled = False   # track whether recession legend entry has been added
for s, e in zip(rec_starts, rec_ends):
    ax.axvspan(s, e, alpha=0.15, color='grey', label='NBER Recession' if not labeled else '')
    labeled = True

ax.set_title('US CPI Inflation — Headline vs Core (Year-over-Year %)', fontsize=14, fontweight='bold')
ax.set_xlabel('Date', fontsize=12)
ax.set_ylabel('Year-over-Year Change (%)', fontsize=12)
ax.legend(fontsize=10)
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print()
print('Discussion:')
print('  When did headline CPI peak? When did the Fed begin hiking rates?')
print('  Was the Fed ahead of or behind the inflation surge?')
print('  Why does core CPI tend to be stickier (slower to fall) than headline?')

In [ ]:
# ── REAL RETURN CALCULATION ────────────────────────────────────────────────────
# We already have the GIS portfolio's annualised nominal return from above.
# Re-compute it here for clarity if it is not in scope from the data download.

# Download GIS prices (re-download to keep this section self-contained)
ALL_TICKERS  = ['AAPL', 'MSFT', 'JPM', 'XOM', 'JNJ', 'GLD', 'IEF']   # 7 GIS holdings
gis_end      = date.today()
gis_start    = date(gis_end.year - 3, gis_end.month, gis_end.day)   # 3 years of price history

gis_raw    = yf.download(
    tickers     = ALL_TICKERS,
    start       = gis_start,
    end         = gis_end,
    auto_adjust = True,      # adjusts for splits and dividends automatically
    progress    = False,     # suppress progress bar — keeps output clean
)
gis_prices = gis_raw['Close'].ffill()   # extract close prices and forward-fill any gaps

gis_returns_daily = (
    gis_prices
    .pct_change()   # day-over-day percentage change for each holding
    .dropna()       # drop first row (NaN — no prior day)
)

n_assets    = len(ALL_TICKERS)                       # 7 holdings
weights_gis = np.repeat(1/n_assets, n_assets)        # equal weight: 1/7 each

portfolio_return_daily = gis_returns_daily.dot(weights_gis)   # weighted daily return
ann_nominal_return     = portfolio_return_daily.mean() * 252   # annualise

# Current annual CPI rate (latest available YoY value, expressed as decimal)
current_cpi_yoy = cpi_headline_yoy.iloc[-1] / 100    # convert from % back to decimal

# Real return = (1 + nominal) / (1 + inflation) − 1
# Simplified approximation: real ≈ nominal − inflation (valid for small values)
real_return_approx = ann_nominal_return - current_cpi_yoy   # the simple approximation
real_return_exact  = (1 + ann_nominal_return) / (1 + current_cpi_yoy) - 1   # the exact formula

print('=== REAL RETURN CALCULATION ===')
print(f'  Annualised Nominal Return (GIS): {ann_nominal_return:.2%}')
print(f'  Current Headline CPI (YoY):      {current_cpi_yoy:.2%}')
print()
print(f'  Real Return (approximation):  {real_return_approx:.2%}')
print(f'  Real Return (exact formula):  {real_return_exact:.2%}')
print()
if real_return_exact > 0:
    print(f'The GIS portfolio has a POSITIVE real return of {real_return_exact:.2%}.')
    print('Investors are ahead of inflation.')
else:
    print(f'The GIS portfolio has a NEGATIVE real return of {real_return_exact:.2%}.')
    print('Despite positive nominal gains, inflation has eroded the real value of the portfolio.')
print()
print('Note: this is a backward-looking calculation using historical returns.')
print('Forward-looking real returns depend on future inflation — which is uncertain.')

In [ ]:
# ── OPTIONAL: EXPORT REAL RETURNS TO CSV ─────────────────────────────────────
# Build a real returns summary DataFrame for export or presentation

real_returns_summary = pd.DataFrame({
    'Metric': [
        'Annualised Nominal Return (GIS, 3Y)',
        'Current Headline CPI YoY',
        'Current Core CPI YoY',
        'Real Return (exact)',
        'Real Return (approximation)',
    ],
    'Value (%)': [
        round(ann_nominal_return * 100, 2),
        round(current_cpi_yoy * 100, 2),
        round(cpi_core_yoy.iloc[-1], 2),
        round(real_return_exact * 100, 2),
        round(real_return_approx * 100, 2),
    ]
})

print('Real Returns Summary:')
print(real_returns_summary.to_string(index=False))
print()

# real_returns_summary.to_csv('gis_real_returns.csv', index=False)   # ← uncomment to save
print('Optional: uncomment the line above to save real returns to CSV.')

---
## Section 3: FX Dynamics

### Why FX matters for a multi-currency portfolio

Currency movements are **invisible costs or gains** that sit between the local asset return and what the investor actually receives in their base currency. A UK equity position returning +10% in GBP terms becomes a 0% return in USD terms if GBP falls 10% against the USD.

**The decomposition:**

$$\text{USD Return} = (1 + \text{Local Return}) \times (1 + \text{FX Return}) - 1$$

For small numbers, this simplifies to: **USD Return ≈ Local Return + FX Return**

This means FX is an **additive component** of total return — it can help (tailwind) or hurt (headwind) the portfolio.

### Currency conventions: base vs. quote

FRED's naming conventions for FX rates:
- `DEXUSUK` = **USD per GBP** — "how many dollars does one pound buy?" (1 GBP = 1.27 USD). A **fall** in this number means GBP is weakening against the dollar.
- `DEXUSEU` = **USD per EUR** — same convention. A fall = EUR weakening.
- `DEXJPUS` = **JPY per USD** — **inverted** convention. A **rise** means the dollar is strengthening (more yen per dollar = yen is weakening). We flip it to USD per JPY for consistency.

In Bloomberg, FX rates are displayed as `GBP Curncy`, `EUR Curncy`, `JPY Curncy`.

### The GIS portfolio and FX

The current GIS portfolio holds **US-listed assets** (AAPL, MSFT, JPM, XOM, JNJ, GLD, IEF). All are priced in USD, so there is **no direct FX translation risk**.

However:
1. **Indirect FX exposure**: Apple earns ~60% of its revenue outside the US. A strong dollar reduces the USD value of those foreign revenues — hurting earnings and, eventually, the stock price.
2. **Portfolio expansion**: GIS invests globally. Adding HSBC (GBP), LVMH (EUR), or Toyota (JPY) introduces direct FX exposure.
3. **Currency hedging**: institutional investors with international exposure often hedge part of the FX risk using forward contracts. This session introduces the concept — the hedging mechanics are covered in Sessions 5–6.

### The September 2022 UK Mini-Budget: a case study

On 23 September 2022, then-UK Chancellor Kwasi Kwarteng announced an unfunded tax-cutting budget. Markets reacted instantly: GBP/USD fell from ~1.13 to 1.035 — a 9% decline in three days (briefly touching a historic low of 1.035).

For a USD investor holding UK equities at the time:
- UK equities also fell ~5% in GBP terms that week
- FX added another ~9% loss
- Total USD return: approximately -14% in one week from a combination of equity and FX shock

This is why currency risk management is not optional for international portfolios — it can dominate the equity signal in extreme events.

In [ ]:
# ── FX RATES FROM FRED ────────────────────────────────────────────────────────
# FRED convention:
#   DEXUSUK: USD per GBP — 'how many dollars does one pound buy?'
#   DEXUSEU: USD per EUR — 'how many dollars does one euro buy?'
#   DEXJPUS: JPY per USD — NOTE: this is INVERTED — 'how many yen does one dollar buy?'
#            We invert it to get USD per JPY for consistency.

fx_gbp = fred.get_series(
    'DEXUSUK',                     # USD per GBP
    observation_start=start_date,
    observation_end=end_date,
)   # a fall in this rate means GBP weakening vs USD

fx_eur = fred.get_series(
    'DEXUSEU',                     # USD per EUR
    observation_start=start_date,
    observation_end=end_date,
)   # a fall means EUR weakening vs USD

fx_jpy_raw = fred.get_series(
    'DEXJPUS',                     # JPY per USD (inverted convention)
    observation_start=start_date,
    observation_end=end_date,
)   # a RISE in this rate means JPY weakening (more yen per dollar)

fx_jpy = 1 / fx_jpy_raw   # invert to get USD per JPY — consistent with GBP and EUR

print('=== FX DATA ===')
print(f'  USD/GBP current: {fx_gbp.dropna().iloc[-1]:.4f}  (1 GBP = {fx_gbp.dropna().iloc[-1]:.4f} USD)')
print(f'  USD/EUR current: {fx_eur.dropna().iloc[-1]:.4f}  (1 EUR = {fx_eur.dropna().iloc[-1]:.4f} USD)')
print(f'  USD/JPY current: {fx_jpy.dropna().iloc[-1]:.6f}  (1 JPY = {fx_jpy.dropna().iloc[-1]:.6f} USD)')
print()
print(f'  Observations (GBP): {len(fx_gbp.dropna())}')
print(f'  Observations (EUR): {len(fx_eur.dropna())}')
print(f'  Observations (JPY): {len(fx_jpy.dropna())}')

In [ ]:
# ── USD/GBP TIME SERIES WITH MINI-BUDGET ANNOTATION ──────────────────────────
fx_gbp_clean = fx_gbp.dropna()   # remove any NaN rows (weekends / holidays)

fig, ax = plt.subplots(figsize=(14, 5))

ax.plot(
    fx_gbp_clean.index,
    fx_gbp_clean.values,
    color='steelblue',
    linewidth=2,
    label='USD/GBP exchange rate',
)

# Annotate the Sep 2022 mini-budget GBP crash
# Find the minimum value in the Sep–Oct 2022 window (the crash trough)
window_start_2022 = '2022-09-01'
window_end_2022   = '2022-11-01'

# Only annotate if this window falls within our data range
if window_start_2022 in fx_gbp_clean.index.astype(str) or fx_gbp_clean.index[0].year <= 2022:
    try:
        window_2022 = fx_gbp_clean[window_start_2022:window_end_2022]   # slice to Sep–Oct 2022
        if len(window_2022) > 0:   # only annotate if data exists in that window
            min_date_2022 = window_2022.idxmin()   # date of the lowest GBP value
            min_val_2022  = window_2022.min()       # the lowest GBP/USD value

            ax.annotate(
                'UK Mini-Budget\nSep 2022\n(GBP flash crash)',
                xy=(min_date_2022, min_val_2022),                     # point to the trough
                xytext=(min_date_2022, min_val_2022 + 0.04),          # label above the trough
                arrowprops=dict(arrowstyle='->', color='crimson'),
                fontsize=9,
                color='crimson',
                ha='center',
            )
    except Exception:
        pass   # if the window is outside the data range, skip the annotation silently

ax.set_title('USD/GBP Exchange Rate — 1 GBP = x USD', fontsize=14, fontweight='bold')
ax.set_xlabel('Date', fontsize=12)
ax.set_ylabel('USD per GBP', fontsize=12)
ax.legend(fontsize=10)
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print()
print('Key observation: a fall in USD/GBP means GBP is weakening — fewer dollars per pound.')
print('For a USD investor holding GBP-denominated assets, this is a headwind:')
print('even if the asset performs well in GBP terms, it translates to fewer USD.')

In [ ]:
# ── FX RETURN DECOMPOSITION ────────────────────────────────────────────────────
# Worked example: a GIS analyst holds a hypothetical UK large-cap equity position.
# The position returned +8% in GBP terms over the past year.
# But what is the USD return, after accounting for GBP/USD movement?

# Decomposition formula:
#   USD Return ≈ Local Currency Return + FX Return
# More precisely:
#   (1 + USD Return) = (1 + Local Return) × (1 + FX Return)
#   USD Return = (1 + Local Return) × (1 + FX Return) − 1

# Compute the 1-year GBP/USD return (positive = GBP strengthened vs USD)
# Use 252 trading days back; fall back to the earliest available if shorter
lookback = min(252, len(fx_gbp_clean) - 1)    # number of trading days to look back
fx_gbp_1y_ago  = fx_gbp_clean.iloc[-lookback - 1]   # GBP/USD rate 1 year ago
fx_gbp_current = fx_gbp_clean.iloc[-1]              # GBP/USD rate today
fx_return_gbp  = (fx_gbp_current / fx_gbp_1y_ago) - 1   # GBP appreciation vs USD

local_return = 0.08   # hypothetical: the UK equity returned +8% in GBP terms

# Apply the exact decomposition formula
usd_return = (1 + local_return) * (1 + fx_return_gbp) - 1   # total USD return

print('=== FX-ADJUSTED RETURN DECOMPOSITION ===')
print(f'Hypothetical UK equity return (in GBP): {local_return:.2%}')
print(f'GBP/USD change over the past year:      {fx_return_gbp:+.2%}')
print(f'USD total return:                        {usd_return:+.2%}')
print()
print(f'GBP {"strengthened" if fx_return_gbp > 0 else "weakened"} by {abs(fx_return_gbp):.2%} vs USD.')
if fx_return_gbp < 0:
    print(f'This FX headwind reduced a {local_return:.0%} GBP return to {usd_return:.2%} in USD terms.')
else:
    print(f'This FX tailwind boosted a {local_return:.0%} GBP return to {usd_return:.2%} in USD terms.')
print()

# ── STACKED BAR CHART SHOWING THE DECOMPOSITION ───────────────────────────────
fig, ax = plt.subplots(figsize=(6, 5))

components = ['Local (GBP)\nReturn', 'FX Effect\n(GBP/USD)', 'Total USD\nReturn']   # bar labels
values     = [local_return, fx_return_gbp, usd_return]                               # bar heights
colours    = ['steelblue' if v >= 0 else 'crimson' for v in values]                  # colour coding

bars = ax.bar(
    components,
    [v * 100 for v in values],   # convert to percentage for readability
    color=colours,
    edgecolor='white',
    width=0.5,
)

ax.axhline(0, color='black', linewidth=1)   # zero reference line
ax.set_title('FX Return Decomposition — Hypothetical UK Equity', fontsize=12, fontweight='bold')
ax.set_ylabel('Return (%)', fontsize=11)

# Label each bar with its return value
for bar, val in zip(bars, values):
    ax.text(
        bar.get_x() + bar.get_width() / 2,     # centre of the bar
        bar.get_height() + (0.2 if val >= 0 else -0.5),   # just outside bar tip
        f'{val:.2%}',
        ha='center',
        va='bottom' if val >= 0 else 'top',
        fontsize=10,
        fontweight='bold',
    )

plt.tight_layout()
plt.show()

---
## Section 4: Macro Dashboard

### The investment use case: the morning brief

Every investment team at a major asset manager starts the day with a **morning brief** — a single page (or screen) capturing the state of the key macro indicators before markets open. It typically covers:
- Overnight equity market moves
- Rates and credit spreads
- Inflation and central bank positioning
- Currency moves
- Any scheduled data releases for the day

The goal is not to react to every data point — it is to **spot deviations from the trend** quickly. An analyst who has seen the same dashboard for 200 mornings in a row will immediately notice when something is out of place.

In Bloomberg, this is typically a custom `BSCREEN` or `FLDS` layout. In this notebook, we assemble it programmatically — the same indicators, updated automatically every time the notebook runs.

### The regime classification framework

A **macro regime** is a label for the current economic environment based on a small set of observable indicators. It is not a forecast — it is a **framework for discussion**.

The four classical regimes (derived from the Bridgewater Associates framework):

| Regime | Growth | Inflation | Typical asset class winners |
|---|---|---|---|
| Expansion | Rising | Contained | Equities, Credit, EM |
| Stagflation | Falling | Rising | Gold, Commodities, TIPS |
| Recession | Falling | Falling | Bonds, Cash, Defensive Equity |
| Recovery | Rising | Falling | Equities, Credit, HY |

The classification below uses three observable indicators: CPI (inflation signal), the 2Y/10Y spread (growth/recession signal), and the unemployment rate (labour market condition). It is intentionally simplified — real regime models use 20+ indicators and machine learning classifiers — but it provides a useful starting point for discussion.

**Important caveat:** regime labels should never mechanically dictate asset allocation. They are a starting point for questions, not answers.

In [ ]:
# ── MACRO DASHBOARD ────────────────────────────────────────────────────────────
# Assemble a single-row DataFrame capturing the current value of 5 key macro indicators.
# This is the foundation of a 'morning brief' dashboard — updated daily, read before markets open.

# Download Fed Funds Effective Rate (daily)
fed_funds = fred.get_series(
    'DFF',                             # Federal Funds Effective Rate (daily)
    observation_start=start_date,
    observation_end=end_date,
)

# Download Unemployment Rate (monthly)
unemployment = fred.get_series(
    'UNRATE',                          # US Unemployment Rate (monthly, seasonally adjusted)
    observation_start=start_date,
    observation_end=end_date,
)

# Assemble current values of each indicator
# Each .iloc[-1] gives the most recent available reading
dashboard = pd.DataFrame({
    'Indicator': [
        'CPI (Headline YoY %)',
        'CPI (Core YoY %)',
        'Fed Funds Rate (%)',
        '2Y/10Y Spread (pp)',
        'Unemployment Rate (%)',
        'USD/EUR',
        'USD/GBP',
    ],
    'Current Value': [
        round(cpi_headline_yoy.dropna().iloc[-1], 2),   # latest headline CPI YoY %
        round(cpi_core_yoy.dropna().iloc[-1], 2),       # latest core CPI YoY %
        round(fed_funds.dropna().iloc[-1], 2),          # latest Fed Funds rate %
        round(spread_series.dropna().iloc[-1], 2),      # latest 2Y/10Y spread pp
        round(unemployment.dropna().iloc[-1], 2),       # latest unemployment rate %
        round(fx_eur.dropna().iloc[-1], 4),             # latest USD/EUR rate
        round(fx_gbp_clean.iloc[-1], 4),                # latest USD/GBP rate
    ],
    'Prior Reading': [
        round(cpi_headline_yoy.dropna().iloc[-2], 2),   # one reading before latest
        round(cpi_core_yoy.dropna().iloc[-2], 2),
        round(fed_funds.dropna().iloc[-2], 2),
        round(spread_series.dropna().iloc[-2], 2),
        round(unemployment.dropna().iloc[-2], 2),
        round(fx_eur.dropna().iloc[-2], 4),
        round(fx_gbp_clean.iloc[-2], 4),
    ],
})

# Add a change column — current minus prior
dashboard['Change'] = (dashboard['Current Value'] - dashboard['Prior Reading']).round(4)

# Add a direction arrow for quick visual scanning
dashboard['Direction'] = dashboard['Change'].apply(
    lambda x: '▲' if x > 0 else ('▼' if x < 0 else '→')   # up / down / flat
)

print('=== MACRO DASHBOARD ===')
print(f'As of: {date.today()}')
print()
print(dashboard.to_string(index=False))

In [ ]:
# ── MACRO REGIME CLASSIFICATION ────────────────────────────────────────────────
# A simple rule-based system to label the current environment.
# This is a framework for discussion — not a forecast or an investment recommendation.
# Each regime has different historical implications for asset class performance.

current_cpi    = cpi_headline_yoy.dropna().iloc[-1]    # latest headline CPI YoY %
current_spread = spread_series.dropna().iloc[-1]       # latest 2Y/10Y spread pp
current_unemp  = unemployment.dropna().iloc[-1]        # latest unemployment rate %

# Prior readings — use 12 months ago for unemployment direction
# Handle gracefully if we have fewer than 13 monthly readings
unemp_clean    = unemployment.dropna()
unemp_12m_ago  = unemp_clean.iloc[-13] if len(unemp_clean) >= 13 else unemp_clean.iloc[0]
unemp_rising   = current_unemp > unemp_12m_ago   # True if unemployment has risen over past year

# ── RULE-BASED REGIME CLASSIFICATION ─────────────────────────────────────────
# The rules below encode a simplified version of the four-quadrant macro framework.
# Priority order: stagflation warning → late cycle → expansion → recovery → mixed signals

if current_cpi > 4.0 and current_spread < 0:
    # High inflation AND inverted yield curve = stagflation warning
    regime = 'STAGFLATION WARNING'
    regime_note = 'High inflation + inverted yield curve. Historically challenging for both equities and bonds.'

elif current_spread < 0 and unemp_rising:
    # Inverted curve AND rising unemployment = late cycle / recession risk
    regime = 'LATE CYCLE / RECESSION RISK'
    regime_note = 'Inverted curve + rising unemployment. Monitor credit spreads and earnings revisions closely.'

elif current_cpi < 3.0 and current_spread > 0 and not unemp_rising:
    # Normal curve + contained inflation + stable employment = healthy expansion
    regime = 'EXPANSION'
    regime_note = 'Normal curve, contained inflation, stable employment. Typically supportive of risk assets.'

elif unemp_rising and current_cpi < 3.0:
    # Employment still adjusting but inflation contained = recovery / early cycle
    regime = 'RECOVERY / EARLY CYCLE'
    regime_note = 'Employment still adjusting but inflation contained. Central bank likely easing.'

else:
    # Indicators are not clearly aligned
    regime = 'MIXED SIGNALS'
    regime_note = 'Indicators are not clearly aligned. Heightened uncertainty — favour diversification.'

print('=== REGIME CLASSIFICATION ===')
print(f'  CPI (YoY):        {current_cpi:.1f}%')
print(f'  2Y/10Y Spread:    {current_spread:+.2f} pp')
print(f'  Unemployment:     {current_unemp:.1f}%  ({"rising" if unemp_rising else "stable/falling"} vs 12M ago)')
print()
print(f'  CURRENT REGIME:   {regime}')
print()
print(f'  Note: {regime_note}')
print()
print('Important caveat: this is a heuristic classification based on 3 indicators.')
print('It is a starting point for discussion, not a portfolio recommendation.')
print('Real regime analysis uses many more indicators and qualitative judgement.')

In [ ]:
# ── OPTIONAL: EXPORT MACRO DATA TO CSV ───────────────────────────────────────
# This cell saves three CSV files: yield curve, real returns summary, and macro dashboard.
# Uncomment the lines you want to activate.

# 1. Full yield curve history (daily rates at 8 maturities)
# yield_curve_df.to_csv('gis_yield_curve_tracker.csv')   # ← uncomment to save

# 2. Real returns summary
# real_returns_summary.to_csv('gis_real_returns.csv', index=False)   # ← uncomment to save

# 3. Macro dashboard snapshot
# dashboard.to_csv('gis_macro_dashboard.csv', index=False)   # ← uncomment to save

# Display the dashboard one more time for a clean end-of-section summary
print('=== MACRO DASHBOARD SNAPSHOT ===')
print(f'Date: {date.today()}')
print(f'Regime: {regime}')
print()
print(dashboard[['Indicator', 'Current Value', 'Direction']].to_string(index=False))
print()
print('Optional exports: uncomment the lines above to save CSV files.')

---
## Discussion

> **"Which of these macro indicators do you currently monitor manually — and how often?"**
>
> **"What would change about your investment process if this dashboard updated automatically every morning before markets open?"**
>
> **"Which indicator would you add to make it more useful for your specific role?"**

Consider:
- A fixed income portfolio manager might want credit spreads (HY OAS, IG OAS) and breakeven inflation rates
- An equity portfolio manager might want earnings revision breadth, the VIX (equity volatility index), or PMI data
- A global macro manager might want commodity prices (WTI crude, copper), emerging market FX, and the dollar index (DXY)

Every indicator can be downloaded from FRED or yfinance using the patterns in this notebook. The `fred.get_series('SERIES_ID')` pattern works for any of the 800,000+ series on the FRED database.

---
## Session 4 Debrief

### What to take away:
- **The yield curve** is a forward-looking signal embedded in market prices — it reflects what bond investors collectively expect for growth and inflation. It is imperfect, but no leading indicator has a better track record over the past 50 years.
- **Inflation erodes real returns.** A portfolio up 5% nominally in a year of 7% inflation has delivered a negative real return. Institutional investors should always state their performance target in real terms.
- **FX is a hidden source of return and risk** in international portfolios. A 10% local return can become a 0% USD return if the currency moves against you. Currency hedging is a strategic decision, not a default.
- **The macro dashboard is a discipline, not a tool.** The value is in the process — looking at the same set of indicators consistently, so that changes stand out clearly.

### Discussion: "Which of these indicators do you currently monitor manually?"
Most investment professionals track some of these — but usually in isolation, from different sources, at different frequencies. The value of this session is to show how to assemble them into a single, consistent picture that updates automatically.

### Key formulas from today

| Concept | Formula |
|---|---|
| CPI year-over-year | $(\text{CPI}_t / \text{CPI}_{t-12}) - 1$ |
| Real return (exact) | $(1 + R_{nominal}) / (1 + \pi) - 1$ |
| Real return (approx) | $R_{nominal} - \pi$ |
| FX-adjusted return | $(1 + R_{local}) \times (1 + R_{FX}) - 1$ |
| 2Y/10Y spread | $R_{10Y} - R_{2Y}$ |

### Preview of Day 3 (tomorrow morning):
*"How do I turn what we have built over two days into interactive, shareable tools — and direct Claude Code to extend them in 90 minutes?"*